In [ ]:
!pip install ucimlrepo

In [ ]:

from ucimlrepo import fetch_ucirepo

# Load regression dataset
dataset = fetch_ucirepo(id=186)

X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (5197, 11)
Testing data: (1300, 11)


In [ ]:

import numpy as np
from sklearn.tree import DecisionTreeRegressor

class GBTRegressor:

    def __init__(self, n_estimators=100, learning_rate=0.1,
                 max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):

        y = np.array(y)
        self.initial_prediction = np.mean(y)

        predictions = np.full(len(y), self.initial_prediction)

        for i in range(self.n_estimators):

            residuals = y - predictions

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth
            )

            tree.fit(X, residuals)

            update = tree.predict(X)

            predictions += self.learning_rate * update

            self.trees.append(tree)

    def predict(self, X):

        predictions = np.full(
            len(X), self.initial_prediction
        )

        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        return predictions

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

gbt = GBTRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

gbt.fit(X_train, y_train)

y_pred = gbt.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("GBT Regressor Results")
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)



GBT Regressor Results
MSE: 0.46292837634267225
RMSE: 0.6803884010935756
R2 Score: 0.3731901479163333


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

sk_gbt = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

sk_gbt.fit(X_train, y_train)

sk_pred = sk_gbt.predict(X_test)

print("Sklearn GBT Results")
print("MSE:", mean_squared_error(y_test, sk_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, sk_pred)))
print("R2 Score:", r2_score(y_test, sk_pred))


Sklearn GBT Results
MSE: 0.46159495765456854
RMSE: 0.6794077992300122
R2 Score: 0.3749956107338409


In [ ]:
#####GBT CLASSIFER


# Load classification dataset https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
dataset = fetch_ucirepo(id=17)
X = dataset.data.features
y = dataset.data.targets
y = y.values.ravel() # flatten to 1D array
y = (y == 'M').astype(int)  # Convert labels to 0 and 1

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (455, 30)
Testing data: (114, 30)


In [ ]:

import numpy as np
from sklearn.tree import DecisionTreeRegressor

class GBTClassifier:

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        y = np.array(y)

        self.trees = []

        p = np.mean(y)
        self.initial_prediction = np.log(p / (1 - p))

        predictions = np.full(len(y), self.initial_prediction)

        for i in range(self.n_estimators):

            probabilities = 1 / (1 + np.exp(-predictions))

            residuals = y - probabilities

            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)

            update = tree.predict(X)

            predictions += self.learning_rate * update

            self.trees.append(tree)

    def predict_proba(self, X):
        predictions = np.full(len(X), self.initial_prediction)

        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        probabilities = 1 / (1 + np.exp(-predictions))

        return probabilities

    def predict(self, X):
        probabilities = self.predict_proba(X)

        return (probabilities >= 0.5).astype(int)



In [ ]:

from sklearn.metrics import accuracy_score, classification_report

gbt_clf = GBTClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

gbt_clf.fit(X_train, y_train)

y_pred = gbt_clf.predict(X_test)

print("Custom GBT Classifier Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Custom GBT Classifier Results
Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        72
           1       1.00      0.88      0.94        42

    accuracy                           0.96       114
   macro avg       0.97      0.94      0.95       114
weighted avg       0.96      0.96      0.96       114



In [ ]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

sk_gbt_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

sk_gbt_clf.fit(X_train, y_train)

sk_pred = sk_gbt_clf.predict(X_test)

print("Sklearn GBT Classifier Results")
print("Accuracy:", accuracy_score(y_test, sk_pred))
print(classification_report(y_test, sk_pred))



Sklearn GBT Classifier Results
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114



In [ ]:
###XBOOST

!pip install xgboost

In [ ]:
from xgboost import XGBRegressor, XGBClassifier

In [ ]:

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xgb_reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb_reg.fit(X_train, y_train)

y_pred_xgb = xgb_reg.predict(X_test)

print("XGBoost Regressor Results")
print("MSE:", mean_squared_error(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("R2 Score:", r2_score(y_test, y_pred_xgb))

XGBoost Regressor Results
MSE: 0.032766181975603104
RMSE: 0.18101431428371378
R2 Score: 0.8591834306716919


In [ ]:

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric='logloss'
)

xgb_clf.fit(X_train, y_train)

y_pred_xgb = xgb_clf.predict(X_test)

print("XGBoost Classifier Results")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))



XGBoost Classifier Results
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

